In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import numpy as np
import os
import xarray as xr
import xcdat as xc
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm as BM
import pandas as pd
import matplotlib as mpl
import matplotlib.ticker as mticker
import netCDF4
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

In [3]:
from scipy import stats

In [4]:
mpl.rcParams['font.family'] = 'Droid Sans'
mpl.rcParams['font.size'] = 12
# Edit axes parameters
mpl.rcParams['axes.linewidth'] = 1.5
# Tick properties
mpl.rcParams['xtick.major.size'] = 5
mpl.rcParams['xtick.minor.size'] = 3
mpl.rcParams['xtick.major.width'] = 1
mpl.rcParams['xtick.direction'] = 'out'
mpl.rcParams['ytick.major.size'] = 5
mpl.rcParams['ytick.minor.size'] = 3
mpl.rcParams['ytick.major.width'] = 1
mpl.rcParams['ytick.direction'] = 'out'

In [5]:
# plt.style.use('dark_background')

### Functions needed for the analysis

In [6]:
import matplotlib as m
from matplotlib.colors import BoundaryNorm as BM
import matplotlib.patches as mpatches

def plot_background(ax):
    ax.add_feature(cfeature.COASTLINE, alpha=0.9, lw=1.1)
    ax.set_global()
    # ax.add_feature(cfeature.LAND, color='lightgray')
    # ax.add_feature(cfeature.OCEAN, color='lightgray')
    gl = ax.gridlines(draw_labels=True,
                      linewidth=1, color='gray', alpha=0.01, linestyle='--')
    gl.top_labels = False
    # gl.left_labels = False
    # gl.bottom_labels = False
    gl.right_labels = False
    gl.xlines = False
    # gl.xlocator = mticker.FixedLocator([-180, -45, 0, 45, 180])
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    gl.xlabel_style = {'size': 10, 'color': 'k'}
    gl.ylabel_style = {'size': 10, 'color': 'k'}
    return ax


def plot_maps(x, y, z, titles, labels, cmap, levels, cbar_label = 'Precip', pval = [], nrows=1, ncols=3, figsize=(12,4), land_mask_list = [0], add_patch=False):
    fig, axarr = plt.subplots(nrows=nrows, ncols=ncols, figsize=figsize, constrained_layout=True, subplot_kw={'projection':ccrs.Robinson(central_longitude=180)})
    
    axlist = axarr.flatten()
    
    for ax in axlist:
        plot_background(ax)
    
    for i in range(len(z)):
        axlist[i].contourf(x, y, z[i], cmap = cmap, transform = ccrs.PlateCarree(central_longitude=0), levels=levels, extend='both')
        axlist[i].set_title(titles[i])
        if i in land_mask_list:
            axlist[i].add_feature(cfeature.LAND, color = 'k', zorder=1)
        if pval != []:
            pval_plot = np.ma.masked_less_equal(pval[i], 0.05)
            axlist[i].pcolor(x, y, pval_plot, alpha = 0., hatch='////', transform = ccrs.PlateCarree(central_longitude=0))
        axlist[i].set_title(titles[i], fontdict={'fontsize':12})
        axlist[i].text(0.1, 1.05, labels[i], size=16, fontweight='bold', transform=axlist[i].transAxes)
        if add_patch:
            axlist[i].add_patch(mpatches.Rectangle(xy=[120, -65], width=170, height=20,
                                            facecolor='none', edgecolor='k',
                                            transform=ccrs.PlateCarree()))
            axlist[i].add_patch(mpatches.Rectangle(xy=[190, -5], width=80, height=10,
                                            facecolor='none', edgecolor='k',
                                            transform=ccrs.PlateCarree()))
            axlist[i].add_patch(mpatches.Rectangle(xy=[140, -5], width=30, height=10,
                                            facecolor='none', edgecolor='k',
                                            transform=ccrs.PlateCarree()))
            axlist[i].add_patch(mpatches.Rectangle(xy=[250, -30], width=40, height=20,
                                            facecolor='none', edgecolor='k',
                                            transform=ccrs.PlateCarree()))
        
    norm = BM(levels, 256, extend='both')
    fig.colorbar(m.cm.ScalarMappable(norm = norm, cmap=cmap), ax = axlist, \
                orientation = 'horizontal', shrink=0.4, aspect = 20, pad = 0.05, label = cbar_label)

In [7]:
from functions import preproc_funcs as funcs

In [8]:
from functions import xr_lowess

In [9]:
from statsmodels.tsa.seasonal import STL


def loess1d(x, period):
    x_copy = x.copy()
    res = STL(x_copy, period=period).fit()
    return res.trend


def loess3d(x, dim, period):
    return xr.apply_ufunc(loess1d, x, input_core_dims=[[dim]], output_core_dims=[[dim]], kwargs=dict(period=period), vectorize=True, dask="parallelized")

In [10]:
import glob
import multiprocessing as mp

In [11]:
# Function to find the first file in each model's r1* directory
def find_all_files(pattern):
    all_paths = glob.glob(pattern)
    model_files = {}
    for path in all_paths:
        # Adjust the split indices based on your folder structure
        path_parts = path.split('/')
        # Assuming the model name is at index 7 (adjust if needed)
        model_identifier = path_parts[7]
        if model_identifier not in model_files:
            model_files[model_identifier] = '/'.join(path_parts[:-1]) + '/*.nc'  # Store only the first file for each model
    return model_files


# Function to find the first file in each model's r1* directory
def find_all_files_extended(pattern):
    all_paths = glob.glob(pattern)
    model_files = {}
    for path in all_paths:
        # Adjust the split indices based on your folder structure
        path_parts = path.split('/')
        # Assuming the model name is at index 7 (adjust if needed)
        model_identifier = path_parts[8] + '_' + path_parts[10][:-1]
        if model_identifier not in model_files:
            model_files[model_identifier] = '/'.join(path_parts[:-1]) + '/*.nc'  # Store only the first file for each model
    return model_files



In [12]:
ts_pattern_hist = '/g/data/lp01/CMIP6/CMIP/*/*/historical/r1i1*/Amon/ts/gr1.5/*/*.nc'
ts_pattern_ssp5 = '/g/data/lp01/CMIP6/ScenarioMIP/*/*/ssp585/r1i1*/Amon/ts/gr1.5/*/*.nc'
ts_pattern_ssp3 = '/g/data/lp01/CMIP6/ScenarioMIP/*/*/ssp370/r1i1*/Amon/ts/gr1.5/*/*.nc'
ts_pattern_ssp2 = '/g/data/lp01/CMIP6/ScenarioMIP/*/*/ssp245/r1i1*/Amon/ts/gr1.5/*/*.nc'
ts_pattern_ssp1 = '/g/data/lp01/CMIP6/ScenarioMIP/*/*/ssp126/r1i1*/Amon/ts/gr1.5/*/*.nc'


In [13]:
ts_pattern_ssp1_ext = '/g/data/*/*/CMIP6/ScenarioMIP/*/*/ssp126/*/Amon/ts/gn/*/*230012.nc'
ts_pattern_ssp5_ext = '/g/data/*/*/CMIP6/ScenarioMIP/*/*/ssp585/*/Amon/ts/gn/*/*230012.nc'
ts_pattern_hist_all = '/g/data/*/*/CMIP6/CMIP/*/*/historical/*/Amon/ts/gn/*/*.nc'
ts_pattern_ssp5_all = '/g/data/*/*/CMIP6/ScenarioMIP/*/*/ssp585/*/Amon/ts/gn/*/*210012.nc'
ts_pattern_ssp5o_ext = '/g/data/*/*/CMIP6/ScenarioMIP/*/*/ssp534-over/*/Amon/ts/gn/*/*230012.nc'

In [ ]:
pr_pattern_hist = '/g/data/lp01/CMIP6/CMIP/*/*/historical/r1i1*/Amon/pr/gr1.5/*/*.nc'
pr_pattern_ssp5 = '/g/data/lp01/CMIP6/ScenarioMIP/*/*/ssp585/r1i1*/Amon/pr/gr1.5/*/*.nc'
pr_pattern_ssp3 = '/g/data/lp01/CMIP6/ScenarioMIP/*/*/ssp370/r1i1*/Amon/pr/gr1.5/*/*.nc'
pr_pattern_ssp2 = '/g/data/lp01/CMIP6/ScenarioMIP/*/*/ssp245/r1i1*/Amon/pr/gr1.5/*/*.nc'
pr_pattern_ssp1 = '/g/data/lp01/CMIP6/ScenarioMIP/*/*/ssp126/r1i1*/Amon/pr/gr1.5/*/*.nc'


In [14]:
psl_pattern_hist = '/g/data/lp01/CMIP6/CMIP/*/*/historical/r1i1*/Amon/psl/gr1.5/*/*.nc'
psl_pattern_ssp5 = '/g/data/lp01/CMIP6/ScenarioMIP/*/*/ssp585/r1i1*/Amon/psl/gr1.5/*/*.nc'
psl_pattern_ssp3 = '/g/data/lp01/CMIP6/ScenarioMIP/*/*/ssp370/r1i1*/Amon/psl/gr1.5/*/*.nc'
psl_pattern_ssp2 = '/g/data/lp01/CMIP6/ScenarioMIP/*/*/ssp245/r1i1*/Amon/psl/gr1.5/*/*.nc'
psl_pattern_ssp1 = '/g/data/lp01/CMIP6/ScenarioMIP/*/*/ssp126/r1i1*/Amon/psl/gr1.5/*/*.nc'


In [15]:
psl_pattern_ssp1_ext = '/g/data/*/*/CMIP6/ScenarioMIP/*/*/ssp126/*/Amon/psl/gn/*/*230012.nc'
psl_pattern_ssp5_ext = '/g/data/*/*/CMIP6/ScenarioMIP/*/*/ssp585/*/Amon/psl/gn/*/*230012.nc'
psl_pattern_hist_all = '/g/data/*/*/CMIP6/CMIP/*/*/historical/*/Amon/psl/gn/*/*.nc'
psl_pattern_ssp5_all = '/g/data/*/*/CMIP6/ScenarioMIP/*/*/ssp585/*/Amon/psl/gn/*/*210012.nc'
psl_pattern_ssp5o_ext = '/g/data/*/*/CMIP6/ScenarioMIP/*/*/ssp534-over/*/Amon/psl/gn/*/*230012.nc'

In [14]:
u10_pattern_hist = '/g/data/*/*/CMIP6/CMIP/*/*/historical/r1i1*/Amon/uas/*/*/*.nc'
u10_pattern_ssp5 = '/g/data/*/*/CMIP6/ScenarioMIP/*/*/ssp585/r1i1*/Amon/uas/*/*/*.nc'
u10_pattern_ssp3 = '/g/data/*/*/CMIP6/ScenarioMIP/*/*/ssp370/r1i1*/Amon/uas/*/*/*.nc'
u10_pattern_ssp2 = '/g/data/*/*/CMIP6/ScenarioMIP/*/*/ssp245/r1i1*/Amon/uas/*/*/*.nc'
u10_pattern_ssp1 = '/g/data/*/*/CMIP6/ScenarioMIP/*/*/ssp126/r1i1*/Amon/uas/*/*/*.nc'


In [15]:
v10_pattern_hist = '/g/data/*/*/CMIP6/CMIP/*/*/historical/r1i1*/Amon/vas/*/*/*.nc'
v10_pattern_ssp5 = '/g/data/*/*/CMIP6/ScenarioMIP/*/*/ssp585/r1i1*/Amon/vas/*/*/*.nc'
v10_pattern_ssp3 = '/g/data/*/*/CMIP6/ScenarioMIP/*/*/ssp370/r1i1*/Amon/vas/*/*/*.nc'
v10_pattern_ssp2 = '/g/data/*/*/CMIP6/ScenarioMIP/*/*/ssp245/r1i1*/Amon/vas/*/*/*.nc'
v10_pattern_ssp1 = '/g/data/*/*/CMIP6/ScenarioMIP/*/*/ssp126/r1i1*/Amon/vas/*/*/*.nc'


In [16]:
msftbarot_pattern_hist = '/g/data/*/*/CMIP6/CMIP/*/*/historical/r1i1*/Omon/msftbarot/*/*/*.nc'
msftbarot_pattern_ssp5 = '/g/data/*/*/CMIP6/ScenarioMIP/*/*/ssp585/r1i1*/Omon/msftbarot/*/*/*.nc'
msftbarot_pattern_ssp3 = '/g/data/*/*/CMIP6/ScenarioMIP/*/*/ssp370/r1i1*/Omon/msftbarot/*/*/*.nc'
msftbarot_pattern_ssp2 = '/g/data/*/*/CMIP6/ScenarioMIP/*/*/ssp245/r1i1*/Omon/msftbarot/*/*/*.nc'
msftbarot_pattern_ssp1 = '/g/data/*/*/CMIP6/ScenarioMIP/*/*/ssp126/r1i1*/Omon/msftbarot/*/*/*.nc'


In [17]:
msftmz_pattern_hist = '/g/data/*/*/CMIP6/CMIP/*/*/historical/r1i1*/Omon/msftmz/*/*/*.nc'
msftmz_pattern_ssp5 = '/g/data/*/*/CMIP6/ScenarioMIP/*/*/ssp585/r1i1*/Omon/msftmz/*/*/*.nc'
msftmz_pattern_ssp3 = '/g/data/*/*/CMIP6/ScenarioMIP/*/*/ssp370/r1i1*/Omon/msftmz/*/*/*.nc'
msftmz_pattern_ssp2 = '/g/data/*/*/CMIP6/ScenarioMIP/*/*/ssp245/r1i1*/Omon/msftmz/*/*/*.nc'
msftmz_pattern_ssp1 = '/g/data/*/*/CMIP6/ScenarioMIP/*/*/ssp126/r1i1*/Omon/msftmz/*/*/*.nc'


In [16]:
ts_files_hist = find_all_files(ts_pattern_hist)
ts_files_ssp5 = find_all_files(ts_pattern_ssp5)
ts_files_ssp3 = find_all_files(ts_pattern_ssp3)
ts_files_ssp2 = find_all_files(ts_pattern_ssp2)
ts_files_ssp1 = find_all_files(ts_pattern_ssp1)

In [17]:
ts_files_ssp1_ext = find_all_files_extended(ts_pattern_ssp1_ext)
ts_files_ssp5_ext = find_all_files_extended(ts_pattern_ssp5_ext)
ts_files_hist_all = find_all_files_extended(ts_pattern_hist_all)
ts_files_ssp5_all = find_all_files_extended(ts_pattern_ssp5_all)
ts_files_ssp5o_ext = find_all_files_extended(ts_pattern_ssp5o_ext)

In [18]:
pr_files_hist = find_all_files(pr_pattern_hist)
pr_files_ssp5 = find_all_files(pr_pattern_ssp5)
pr_files_ssp3 = find_all_files(pr_pattern_ssp3)
pr_files_ssp2 = find_all_files(pr_pattern_ssp2)
pr_files_ssp1 = find_all_files(pr_pattern_ssp1)

In [18]:
psl_files_hist = find_all_files(psl_pattern_hist)
psl_files_ssp5 = find_all_files(psl_pattern_ssp5)
psl_files_ssp3 = find_all_files(psl_pattern_ssp3)
psl_files_ssp2 = find_all_files(psl_pattern_ssp2)
psl_files_ssp1 = find_all_files(psl_pattern_ssp1)

In [19]:
psl_files_ssp1_ext = find_all_files_extended(psl_pattern_ssp1_ext)
psl_files_ssp5_ext = find_all_files_extended(psl_pattern_ssp5_ext)
psl_files_hist_all = find_all_files_extended(psl_pattern_hist_all)
psl_files_ssp5_all = find_all_files_extended(psl_pattern_ssp5_all)
psl_files_ssp5o_ext = find_all_files_extended(psl_pattern_ssp5o_ext)

In [19]:
u10_files_hist = find_all_files(u10_pattern_hist)
u10_files_ssp5 = find_all_files(u10_pattern_ssp5)
u10_files_ssp3 = find_all_files(u10_pattern_ssp3)
u10_files_ssp2 = find_all_files(u10_pattern_ssp2)
u10_files_ssp1 = find_all_files(u10_pattern_ssp1)

In [20]:
v10_files_hist = find_all_files(v10_pattern_hist)
v10_files_ssp5 = find_all_files(v10_pattern_ssp5)
v10_files_ssp3 = find_all_files(v10_pattern_ssp3)
v10_files_ssp2 = find_all_files(v10_pattern_ssp2)
v10_files_ssp1 = find_all_files(v10_pattern_ssp1)

In [26]:
msftbarot_files_hist = find_all_files(msftbarot_pattern_hist)
msftbarot_files_ssp5 = find_all_files(msftbarot_pattern_ssp5)
msftbarot_files_ssp3 = find_all_files(msftbarot_pattern_ssp3)
msftbarot_files_ssp2 = find_all_files(msftbarot_pattern_ssp2)
msftbarot_files_ssp1 = find_all_files(msftbarot_pattern_ssp1)

In [27]:
msftmz_files_hist = find_all_files(msftmz_pattern_hist)
msftmz_files_ssp5 = find_all_files(msftmz_pattern_ssp5)
msftmz_files_ssp3 = find_all_files(msftmz_pattern_ssp3)
msftmz_files_ssp2 = find_all_files(msftmz_pattern_ssp2)
msftmz_files_ssp1 = find_all_files(msftmz_pattern_ssp1)

In [22]:
# paths = ['/g/data/fs38/publications/CMIP6/ScenarioMIP/CSIRO-ARCCSS/ACCESS-CM2/ssp534-over/r1i1p1f1/Amon/ts/gn/v20210928/ts_Amon_ACCESS-CM2_ssp534-over_r1i1p1f1_gn_204101-210012.nc',
# '/g/data/fs38/publications/CMIP6/ScenarioMIP/CSIRO/ACCESS-ESM1-5/ssp534-over/r1i1p1f1/Amon/ts/gn/v20211020/ts_Amon_ACCESS-ESM1-5_ssp534-over_r1i1p1f1_gn_204101-210012.nc',
# '/g/data/oi10/replicas/CMIP6/ScenarioMIP/CAS/FGOALS-g3/ssp534-over/r1i1p1f1/Amon/ts/gn/v20200410/*.nc',
# '/g/data/oi10/replicas/CMIP6/ScenarioMIP/CCCma/CanESM5/ssp534-over/r1i1p1f1/Amon/ts/gn/v20190429/*.nc',
# '/g/data/oi10/replicas/CMIP6/ScenarioMIP/Its/Its-CM6A-LR/ssp534-over/r1i1p1f1/Amon/ts/gr/v20190909/*.nc',
# '/g/data/oi10/replicas/CMIP6/ScenarioMIP/MIROC/MIROC6/ssp534-over/r1i1p1f1/Amon/ts/gn/v20190807/ts_Amon_MIROC6_ssp534-over_r1i1p1f1_gn_204001-210012.nc',
# '/g/data/oi10/replicas/CMIP6/ScenarioMIP/MRI/MRI-ESM2-0/ssp534-over/r1i1p1f1/Amon/ts/gn/v20191108/*.nc',]
# model_names = ['ACCESS-CM2', 'ACCESS-ESM1-5', 'FGOALS-g3', 'CanESM5', 'Its-CM6A-LR', 'MIROC6', 'MRI-ESM2-0']
# ts_files_ssp5_over = dict(zip(model_names, paths))
# ts_files_ssp5_over

In [23]:
# ts_pattern_access_hist = '/g/data/ob22/as8561/data/enso_trans_stable/hist_ssp_runs/regrid_sst/'
# ts_pattern_access_ssp = '/g/data/ob22/as8561/data/enso_trans_stable/hist_ssp_runs/regrid_sst'

In [20]:
import xesmf as xe

In [21]:
# temp_hist = xr.open_dataset(ts_files_hist['ACCESS-CM2'])
# temp_hist

In [22]:
ds_out = xe.util.cf_grid_2d(-0.75, 360, 1.5, -90, 90, 1.5)
ds_out

<xarray.Dataset>
Dimensions:             (lon: 240, bound: 2, lat: 120)
Coordinates:
  * lon                 (lon) float64 0.0 1.5 3.0 4.5 ... 355.5 357.0 358.5
  * lat                 (lat) float64 -89.25 -87.75 -86.25 ... 86.25 87.75 89.25
    latitude_longitude  float64 nan
Dimensions without coordinates: bound
Data variables:
    lon_bounds          (lon, bound) float64 -0.75 0.75 0.75 ... 357.8 359.2
    lat_bounds          (lat, bound) float64 -90.0 -88.5 -88.5 ... 88.5 90.0

In [23]:
from dask.diagnostics import ProgressBar

In [76]:
# Function to process a single model and return the detrended NINO3.4 and precip anomalies
def process_model(model_identifier):
    try:
        # print(f"Processing model: {model_identifier}")
        # Load datasets
        var_file_hist = psl_files_hist_all[model_identifier]
        var_file_ssp = psl_files_ssp1_ext[model_identifier]
        ds_var_hist = xc.open_mfdataset(var_file_hist, use_cftime=True).sel(time = slice('1850-01-01', '2015-01-01'))
        ds_var_ssp = xc.open_mfdataset(var_file_ssp, use_cftime=True)
        # add custom time ranges
        ds_var_hist['time'] = xr.cftime_range('1850-01-01', '2015-01-01', freq='1M')
        ssp_end_year = int(ds_var_ssp.time.dt.year[-1])
        ds_var_ssp['time'] = xr.cftime_range('2015-01-01', f'{ssp_end_year + 1}-01-01', freq='1M')
        combined = xr.concat([ds_var_hist, ds_var_ssp], dim='time')
        regridder = xe.Regridder(combined, ds_out, 'bilinear', periodic=True, ignore_degenerate=True)
        #
        # with ProgressBar():
        var = regridder(combined.psl.resample(time = 'AS-JUN').mean('time')).load()  # var data
        # var = xr.concat([ds_var_hist, ds_var_ssp], dim='time').psl.resample(time = 'AS-JUN').mean('time').load()  # var data
        # precip = ds_precip['pr'] * 86400  # Convert kg/m²/s to mm/day

        # Calculate 3d values
        var_anom = funcs.calc_anom_annual(var, var.sel(time = slice('1960', '1990')))
        # var_trend = funcs.calc_trend3d(var_anom.sel(time = slice('1980', '2014')), 'time')
        # var_trend_pval = funcs.calc_trend_pval3d(var_anom.sel(time = slice('1980', '2014')), 'time')

        # calc timeseries values
        # weights = np.cos(np.deg2rad(var.lat))

        # print(f'Completed: {model_identifie}')
        # return model_identifier, var_trend, var_trend_pval, gmst_anom, nino34_index, wp_var, ct_var, so_var
        return model_identifier, var_anom
    except Exception as e:
        print(f"Error processing {model_identifier}: {e}")



# TODO : remapping required for ssp534_over
def process_model_overshoot(model_identifier):
    try:
        # print(f"Processing model: {model_identifier}")
        # Load datasets
        var_file_hist = psl_files_hist_all[model_identifier]
        var_file_ssp5_over = psl_files_ssp5o_ext[model_identifier]
        var_file_ssp5 = psl_files_ssp5_all[model_identifier]
        ds_var_hist = xr.open_mfdataset(var_file_hist, use_cftime=True).sel(time = slice('1850-01-01', '2015-01-01'))
        ds_var_ssp5_over = xr.open_mfdataset(var_file_ssp5_over, use_cftime=True)
        ds_var_ssp5 = xr.open_mfdataset(var_file_ssp5, use_cftime=True)
        # add custom time ranges
        ds_var_hist['time'] = xr.cftime_range('1850-01-01', '2015-01-01', freq='1M')
        ssp_start_year = int(ds_var_ssp5_over.time.dt.year[0])
        ssp_end_year = int(ds_var_ssp5_over.time.dt.year[-1])
        ds_var_ssp5 = ds_var_ssp5.sel(time = slice(str(2015), str(ssp_start_year-1)))
        ds_var_ssp5['time'] = xr.cftime_range('2015-01-01', f'{ssp_start_year}-01-01', freq='1M')
        ds_var_ssp5_over['time'] = xr.cftime_range(f'{ssp_start_year}-01-01', f'{ssp_end_year + 1}-01-01', freq='1M')
        combined = xr.concat([ds_var_hist, ds_var_ssp5, ds_var_ssp5_over], dim='time')
        regridder = xe.Regridder(combined, ds_out, 'bilinear', periodic=True)
        
        var = regridder = regridder(combined.psl.resample(time = 'AS-JUN').mean('time')).load()  # var data
        # precip = ds_precip['pr'] * 86400  # Convert kg/m²/s to mm/day

        # Calculate 3d values
        var_anom = funcs.calc_anom_annual(var, var.sel(time = slice('1960', '1990')))
        # # var_trend = funcs.calc_trend3d(var_anom.sel(time = slice('1980', '2014')), 'time')
        # var_trend_pval = funcs.calc_trend_pval3d(var_anom.sel(time = slice('1980', '2014')), 'time')

        # calc timeseries values
        # weights = np.cos(np.deg2rad(var.lat))
        # gmst_anom = var_anom.weighted(weights).mean(('lat', 'lon'))
        # nino34_index = funcs.detrend1d_check(var_anom.sel(lat = slice(-5,5), lon = slice(-170+360, -120+360)).weighted(weights).mean(('lat', 'lon')), period=15)
        # wp_var = var_anom.sel(lat = slice(-5,5), lon = slice(140, 170)).weighted(weights).mean(('lat', 'lon'))
        # ct_var = var_anom.sel(lat = slice(-5,5), lon = slice(190, 270)).weighted(weights).mean(('lat', 'lon'))
        # so_var = var_anom.sel(lat = slice(-65, -45), lon = slice(120, 290)).weighted(weights).mean(('lat', 'lon'))

        # print(f'Completed: {model_identifie}')
        return model_identifier, var_anom
    except Exception as e:
        print(f"Error processing {model_identifier}: {e}")
        # break


# 

In [79]:
# models_to_process = [(model) for model in ts_files_hist if model in ts_files_ssp5 and model not in ['Its-CM5A2-INCA', 'KIOST-ESM', 'CIESM', 'EC-Earth3-Veg']]
models_to_process = [(model) for model in psl_files_hist_all if model in psl_files_ssp1_ext and model not in ['IPSL-CM5A2-INCA']]
# models_to_process = [(model) for model in psl_files_hist_all if model in psl_files_ssp5_ext and model not in ['IPSL-CM5A2-INCA', 'GISS-E2-1-G_r4i1p1f2', 'GISS-E2-1-G_r2i1p1f2']]
# models_to_process = [(model) for model in psl_files_hist_all if model in psl_files_ssp5o_ext and model in psl_files_ssp5_all and model not in ['Its-CM5A2-INCA']]
len(models_to_process), models_to_process

(44,
 ['ACCESS-CM2_r1i1p1f',
  'ACCESS-ESM1-5_r27i1p1f',
  'ACCESS-ESM1-5_r10i1p1f',
  'ACCESS-ESM1-5_r31i1p1f',
  'ACCESS-ESM1-5_r9i1p1f',
  'ACCESS-ESM1-5_r5i1p1f',
  'ACCESS-ESM1-5_r26i1p1f',
  'ACCESS-ESM1-5_r11i1p1f',
  'ACCESS-ESM1-5_r8i1p1f',
  'ACCESS-ESM1-5_r30i1p1f',
  'ACCESS-ESM1-5_r4i1p1f',
  'ACCESS-ESM1-5_r25i1p1f',
  'ACCESS-ESM1-5_r29i1p1f',
  'ACCESS-ESM1-5_r33i1p1f',
  'ACCESS-ESM1-5_r12i1p1f',
  'ACCESS-ESM1-5_r7i1p1f',
  'ACCESS-ESM1-5_r24i1p1f',
  'ACCESS-ESM1-5_r28i1p1f',
  'ACCESS-ESM1-5_r32i1p1f',
  'ACCESS-ESM1-5_r13i1p1f',
  'ACCESS-ESM1-5_r6i1p1f',
  'ACCESS-ESM1-5_r40i1p1f',
  'ACCESS-ESM1-5_r23i1p1f',
  'ACCESS-ESM1-5_r39i1p1f',
  'ACCESS-ESM1-5_r1i1p1f',
  'ACCESS-ESM1-5_r18i1p1f',
  'ACCESS-ESM1-5_r14i1p1f',
  'ACCESS-ESM1-5_r35i1p1f',
  'ACCESS-ESM1-5_r22i1p1f',
  'ACCESS-ESM1-5_r38i1p1f',
  'ACCESS-ESM1-5_r19i1p1f',
  'ACCESS-ESM1-5_r15i1p1f',
  'ACCESS-ESM1-5_r34i1p1f',
  'ACCESS-ESM1-5_r21i1p1f',
  'ACCESS-ESM1-5_r3i1p1f',
  'ACCESS-ESM1-5_r37i1p1f',

In [81]:
# Run multiprocessing and gather results
res_arr = []
with mp.Pool(processes=mp.cpu_count()) as pool:
    i = 0
    for res in pool.imap(process_model, models_to_process):
        res_arr.append(res)
        print(f'Completed {i+1}/{len(models_to_process)}', end='\r')
        i += 1



In [82]:
# models = np.array(res_arr)[:, 0]
# trend = np.array(res_arr)[:, 1]
# trend_pval = np.array(res_arr)[:, 2]
# nino34_index = np.array(res_arr)[:, 3]
# wp_sst = np.array(res_arr)[:, 4]
# cp_sst = np.array(res_arr)[:, 5]
# so_sst = np.array(res_arr)[:, 6]

In [83]:
def process_extensions(arr, get_extensions=True):
    if get_extensions:
        return [da for da in arr if len(da.time) == 452]
    else:
        return [da.sel(time = slice('1850-01-01', '2099-12-01')) for da in arr]

In [84]:
model_list = np.array(res_arr)[:, 0]

In [85]:
np.array(res_arr).shape

(44, 2)

In [86]:
model_list

array(['ACCESS-CM2_r1i1p1f', 'ACCESS-ESM1-5_r27i1p1f',
       'ACCESS-ESM1-5_r10i1p1f', 'ACCESS-ESM1-5_r31i1p1f',
       'ACCESS-ESM1-5_r9i1p1f', 'ACCESS-ESM1-5_r5i1p1f',
       'ACCESS-ESM1-5_r26i1p1f', 'ACCESS-ESM1-5_r11i1p1f',
       'ACCESS-ESM1-5_r8i1p1f', 'ACCESS-ESM1-5_r30i1p1f',
       'ACCESS-ESM1-5_r4i1p1f', 'ACCESS-ESM1-5_r25i1p1f',
       'ACCESS-ESM1-5_r29i1p1f', 'ACCESS-ESM1-5_r33i1p1f',
       'ACCESS-ESM1-5_r12i1p1f', 'ACCESS-ESM1-5_r7i1p1f',
       'ACCESS-ESM1-5_r24i1p1f', 'ACCESS-ESM1-5_r28i1p1f',
       'ACCESS-ESM1-5_r32i1p1f', 'ACCESS-ESM1-5_r13i1p1f',
       'ACCESS-ESM1-5_r6i1p1f', 'ACCESS-ESM1-5_r40i1p1f',
       'ACCESS-ESM1-5_r23i1p1f', 'ACCESS-ESM1-5_r39i1p1f',
       'ACCESS-ESM1-5_r1i1p1f', 'ACCESS-ESM1-5_r18i1p1f',
       'ACCESS-ESM1-5_r14i1p1f', 'ACCESS-ESM1-5_r35i1p1f',
       'ACCESS-ESM1-5_r22i1p1f', 'ACCESS-ESM1-5_r38i1p1f',
       'ACCESS-ESM1-5_r19i1p1f', 'ACCESS-ESM1-5_r15i1p1f',
       'ACCESS-ESM1-5_r34i1p1f', 'ACCESS-ESM1-5_r21i1p1f',
       '

In [87]:
test = np.array(res_arr)[:, 1]
model_extensions = []
for i in range(len(test)):
    da = test[i]
    # print(len(da.time))
    if len(da.time) == 452:
        model_extensions.append(model_list[i])



model_extensions

['ACCESS-CM2_r1i1p1f',
 'ACCESS-ESM1-5_r27i1p1f',
 'ACCESS-ESM1-5_r10i1p1f',
 'ACCESS-ESM1-5_r31i1p1f',
 'ACCESS-ESM1-5_r9i1p1f',
 'ACCESS-ESM1-5_r5i1p1f',
 'ACCESS-ESM1-5_r26i1p1f',
 'ACCESS-ESM1-5_r11i1p1f',
 'ACCESS-ESM1-5_r8i1p1f',
 'ACCESS-ESM1-5_r30i1p1f',
 'ACCESS-ESM1-5_r4i1p1f',
 'ACCESS-ESM1-5_r25i1p1f',
 'ACCESS-ESM1-5_r29i1p1f',
 'ACCESS-ESM1-5_r33i1p1f',
 'ACCESS-ESM1-5_r12i1p1f',
 'ACCESS-ESM1-5_r7i1p1f',
 'ACCESS-ESM1-5_r24i1p1f',
 'ACCESS-ESM1-5_r28i1p1f',
 'ACCESS-ESM1-5_r32i1p1f',
 'ACCESS-ESM1-5_r13i1p1f',
 'ACCESS-ESM1-5_r6i1p1f',
 'ACCESS-ESM1-5_r40i1p1f',
 'ACCESS-ESM1-5_r23i1p1f',
 'ACCESS-ESM1-5_r39i1p1f',
 'ACCESS-ESM1-5_r1i1p1f',
 'ACCESS-ESM1-5_r18i1p1f',
 'ACCESS-ESM1-5_r14i1p1f',
 'ACCESS-ESM1-5_r35i1p1f',
 'ACCESS-ESM1-5_r22i1p1f',
 'ACCESS-ESM1-5_r38i1p1f',
 'ACCESS-ESM1-5_r19i1p1f',
 'ACCESS-ESM1-5_r15i1p1f',
 'ACCESS-ESM1-5_r34i1p1f',
 'ACCESS-ESM1-5_r21i1p1f',
 'ACCESS-ESM1-5_r3i1p1f',
 'ACCESS-ESM1-5_r37i1p1f',
 'ACCESS-ESM1-5_r16i1p1f',
 'ACCESS-ESM1

In [88]:
len(model_list), len(model_extensions)

(44, 43)

In [89]:
len(process_extensions(np.array(res_arr)[:, 1], get_extensions=False))

44

In [90]:
# model_trend = xr.concat(np.array(res_arr)[:, 1], dim=model_list).rename(dict(concat_dim = 'model')).to_dataset(name = 'trend')
# model_pval = xr.concat(np.array(res_arr)[:, 2], dim=model_list).rename(dict(concat_dim = 'model')).to_dataset(name = 'pval')


In [91]:
# model_var = xr.concat(process_extensions(np.array(res_arr)[:, 1], get_extensions=False), dim=model_list, coords='minimal', compat='override').rename(dict(concat_dim = 'model')).to_dataset(name = 'ts')
model_var = xr.concat(np.array(res_arr)[:, 1], dim=model_list, coords='minimal', compat='override').rename(dict(concat_dim = 'model')).to_dataset(name = 'psl')

In [92]:
out = xr.merge([model_var])
out

<xarray.Dataset>
Dimensions:             (time: 652, lon: 240, lat: 120, model: 44)
Coordinates:
  * time                (time) object 1849-06-01 00:00:00 ... 2500-06-01 00:0...
  * lon                 (lon) float64 0.0 1.5 3.0 4.5 ... 355.5 357.0 358.5
  * lat                 (lat) float64 -89.25 -87.75 -86.25 ... 86.25 87.75 89.25
    latitude_longitude  float64 nan
  * model               (model) object 'ACCESS-CM2_r1i1p1f' ... 'GISS-E2-1-H_...
Data variables:
    psl                 (model, time, lat, lon) float32 23.96 22.97 ... -410.3

In [58]:
# model_var = xr.concat(process_extensions(np.array(res_arr)[:, 1], get_extensions=True), dim=model_extensions).rename(dict(concat_dim = 'model')).to_dataset(name = 'ts')

In [59]:
# out_extensions = xr.merge([model_var])
# out_extensions

In [60]:
# out.to_netcdf('/g/data/ob22/as8561/data/enso_trans_stable/sst_grad_res/CMIP6_models/ts_ssp1.nc')
# out_extensions.to_netcdf('/g/data/ob22/as8561/data/enso_trans_stable/sst_grad_res/CMIP6_models/ts_ssp1_ext.nc')

In [93]:
out.to_netcdf('/g/data/ob22/as8561/data/enso_trans_stable/sst_grad_res/CMIP6_models/psl_ssp1_ext.nc')